# 03 · "Idiot Sandwich": TRL DPOTrainer + LoRA persona tuning
**PA2 Task 5 (Sec. 11).** Model `HuggingFaceTB/SmolLM2-360M-Instruct`, data `data/persona_lora/ramsay_style_preferences.json` (220 rows of prompt / chosen / rejected). This is the only task where TRL + PEFT are allowed (Sec. 3). Runtime: ~10-20 min on **one** T4.

Why one GPU: the HF/TRL Trainer would otherwise wrap the model in `DataParallel` across both T4s, which is slower than one GPU for a model this small.

In [ ]:
# =============================================================================
# Step 0a: pin this notebook to a single GPU BEFORE torch is imported
# =============================================================================
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

## Step 0 · Copy the code into Kaggle's working folder  (PA2 Task 1, Sec. 7 + Sec. 5.4)
Notebook settings (right panel): **Accelerator = GPU T4 x2**, **Internet = On** (needs a phone-verified Kaggle account).

The code comes from your uploaded Kaggle Dataset (the `small-model-alignment.zip`). Attach it with **+ Add Input → Datasets → Your Datasets**. This cell finds it automatically, so there are no paths to edit.

In [ ]:
# =============================================================================
# Step 0: copy the code from the attached Kaggle Dataset into /kaggle/working
#   /kaggle/working is the ONLY folder Kaggle keeps when you "Save Version" (PA2 Sec. 5.5)
# =============================================================================
import os, shutil, glob

REPO_DIR = "/kaggle/working/small-model-alignment"

def find_code_dataset():
    # Locate the uploaded code dataset anywhere under /kaggle/input.
    # Attached notebook OUTPUTS also contain a copy of the repo, but those copies
    # have an outputs/ folder; the clean code dataset does not, so prefer it.
    hits = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/submission_adapter.py", recursive=True)]
    clean = [h for h in hits if not os.path.isdir(os.path.join(h, "outputs"))]
    if not (clean or hits):
        raise FileNotFoundError("Code dataset not found: + Add Input -> Datasets -> Your Datasets -> your code dataset")
    return (clean or hits)[0]

if not os.path.exists(os.path.join(REPO_DIR, "submission_adapter.py")):
    src = find_code_dataset()
    print("copying code from", src)
    shutil.copytree(src, REPO_DIR, dirs_exist_ok=True)
    !chmod -R u+w {REPO_DIR}   # /kaggle/input is read-only; make the copy writable
%cd {REPO_DIR}

# Memory/perf environment for the T4s, inherited by every `!python` / `!torchrun` below:
#   expandable_segments -> less CUDA fragmentation with dynamic (per-batch) padding
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

def sh(cmd):
    # Run a shell command, stream its output here, and STOP the notebook if it fails
    # (so a crashed torchrun does not silently continue during Save & Run All).
    get_ipython().system(cmd)
    if get_ipython().user_ns.get("_exit_code", 0) != 0:
        raise RuntimeError(f"command failed: {cmd[:150]}")
!ls

## Step 1 · Install dependencies without touching PyTorch  (PA2 Sec. 5.2)
Kaggle's preinstalled PyTorch already matches the T4 CUDA runtime, so we never reinstall it. `torchao` is removed because old Kaggle builds break PEFT/LoRA loading.
TRL and PEFT are added here for this task.

In [ ]:
# =============================================================================
# Step 1: install the pinned libraries (exactly the manual's safe pattern)
# =============================================================================
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors pytest
!python -m pip install -q -e . --no-deps
!python -m pip install -q "trl>=0.13" "peft>=0.13"
import trl, peft; print("trl", trl.__version__, "| peft", peft.__version__)

## Step 2 · Look at the data and its conversational format
TRL expects `prompt`, `chosen`, `rejected` as lists of `{"role", "content"}` messages; the tokenizer's chat template turns them into text.

In [ ]:
# =============================================================================
# Step 2: inspect a row
# =============================================================================
import json
rows = json.load(open("data/persona_lora/ramsay_style_preferences.json"))
print(len(rows), "rows"); print(json.dumps(rows[0], indent=2))
!python scripts/prepare_data.py --only persona

## Step 3 · Train the LoRA adapter with DPOTrainer
Chosen configuration (justify in the report, Sec. 11): **r = 16, α = 32, dropout 0.05**, targets = all attention + MLP projections, **lr 5e-5** cosine with 10% warmup, batch 4 × accumulation 2 = 8, **3 epochs**, β = 0.1, fp16 autocast on FP32 weights. With `peft_config` given, TRL uses the same network with the adapter switched off as the reference model, so no second model copy is loaded.

In [ ]:
# =============================================================================
# Step 3: LoRA-DPO training
# =============================================================================
sh("python scripts/train_lora_dpo.py --output_dir outputs/lora_dpo --rank 16 --alpha 32 --dropout 0.05 "
   "--lr 5e-5 --epochs 3 --batch_size 4 --grad_accum 2 --beta 0.1")

In [ ]:
# =============================================================================
# Step 4: training log (loss, rewards/accuracies, rewards/margins reported by TRL)
# =============================================================================
import pandas as pd
log = pd.DataFrame(json.load(open("outputs/lora_dpo/log_history.json")))
cols = [c for c in log.columns if c in ("epoch","step","loss","eval_loss","rewards/accuracies","rewards/margins","eval_rewards/accuracies","eval_rewards/margins")]
log[cols]

## Step 5 · Generate on the persona style prompts: base instruct model vs LoRA-DPO
Same decoding for both (greedy, max 256 tokens, repetition penalty 1.1) and the tokenizer's own chat template (the format TRL trained with).

In [ ]:
# =============================================================================
# Step 5: persona generations + blinded pairs file
# =============================================================================
COMMON = "--eval_set eval_sets/persona_style_eval.json --prompt_format chat_template --max_new_tokens 256"
!python scripts/generate.py --model_path HuggingFaceTB/SmolLM2-360M-Instruct --generator smollm2-360m-instruct {COMMON} --output outputs/generations/smol_base_persona.json
!python scripts/generate.py --model_path outputs/lora_dpo/merged --generator smollm2-lora-dpo {COMMON} --output outputs/generations/smol_lora_persona.json
sh("python scripts/make_pairs.py --a outputs/generations/smol_lora_persona.json --a_name lora_dpo "
   "--b outputs/generations/smol_base_persona.json --b_name smol_base --out outputs/eval/lora_vs_base_persona_pairs.jsonl")

In [ ]:
# Side-by-side look: did the style change, and is the advice still correct and safe? (Sec. 11 last line)
base = json.load(open("outputs/generations/smol_base_persona.json")); lora = json.load(open("outputs/generations/smol_lora_persona.json"))
for b, l in list(zip(base, lora))[::4]:
    print(f"[{b['category']}] {b['instruction']}\n  BASE: {b['output'][:300]}\n  LORA: {l['output'][:300]}\n")

## Final step · Preserve the outputs  (PA2 Sec. 5.5)
1. Click **Save Version → Save & Run All (Commit)** (or, if you ran interactively, *Quick Save* keeps the current outputs).
2. After the version finishes, its `/kaggle/working` folder becomes an **Output** you can attach to the next notebook with **Add Input → Your Work → Notebooks**.